In [ ]:
# Paths come from `paths.py`, never from a literal relative to some checkout.
# That module is the only place that knows where the tree lives, and it names
# the environment variables that override each location (FSCORE_DB above all —
# fscore.db is ~1.7 GB and is not kept in the repository).
# Run this notebook from its own directory, src/fscore_vietnam.
import sys, pathlib

HERE = pathlib.Path.cwd()
assert (HERE / "paths.py").exists(), f"run from src/fscore_vietnam (cwd={HERE})"
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from paths import DATA, RESULTS, DB, ensure_dirs, require_db
ensure_dirs()

# Số lượng cổ phiếu: `paid_in_capital / 10.000` so với `ShareAtPeriodEnd`

Notebook **chỉ chẩn đoán**, không ghi đè bất kỳ output nào của pipeline. Nó tồn tại để trả
lời một câu hỏi trước khi sửa `book_to_market_calculation.ipynb`:

> `shares_issued = paid_in_capital / PAR_VALUE_VND` có phải là số cổ phiếu mà market equity
> cần hay không?

Hai đại lượng khác nhau bị gọi chung là "số cổ phiếu":

| | nghĩa | nguồn trong repo |
|---|---|---|
| **phát hành** (issued) | toàn bộ cổ phiếu đã đăng ký, gồm cả cổ phiếu quỹ | `paid_in_capital / 10.000` |
| **lưu hành** (outstanding) | phát hành trừ cổ phiếu quỹ | `fireant_financial_data_general.ShareAtPeriodEnd` |

BM hiện tại đang trộn hai chuẩn: tử số `book_equity` **đã** trừ cổ phiếu quỹ (VAS ghi âm sẵn
trong `owner_equity`), mẫu số thì **chưa**. Notebook đo xem sự trộn đó lớn tới đâu, và
`ShareAtPeriodEnd` có đủ sạch để thay thế không.

**Bốn câu hỏi, theo thứ tự:**

1. `ShareAtPeriodEnd` có phải chỉ là `PaidInCapital/10.000` chép lại? Nếu đúng thì không có
   gì để bàn.
2. Chỗ hai bên lệch nhau có giải thích được bằng cổ phiếu quỹ không?
3. `ShareAtPeriodEnd` có lỗi riêng của nó không — cụ thể là số cổ phiếu hôm nay bị kéo ngược
   về quá khứ, tức look-ahead?
4. Đổi sang số lưu hành thì BM đổi bao nhiêu, và đổi ở đâu?

In [1]:
import sqlite3
from contextlib import closing

import numpy as np
import pandas as pd

from statement_fields import PAR_VALUE_VND


# `book_to_market_calculation.ipynb` trỏ tới "../cafef.db", nhưng file đó hiện là 0 byte ở
# thư mục gốc — dữ liệu thật nằm trong fscore.db. Đây là một lỗi đường dẫn cần sửa riêng.

# Ngưỡng của quy tắc lọc ở mục G. Đặt thành hằng số ở đây để mục G đo được độ nhạy của
# chúng thay vì rải số ma khắp notebook.
TOL_EQUAL = 0.005   # coi là "khớp": lệch tương đối dưới 0,5%
MAX_TREASURY = 0.30 # mức cổ phiếu quỹ lớn nhất còn tin được

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## Dữ liệu vào

Panel BM đã xuất sẵn (`book_to_market_panel.csv`) mang theo `shares_issued`, `book_equity`,
`close` và cờ `has_treasury` — đủ để dựng lại BM theo cách đếm khác mà không phải chạy lại
toàn bộ notebook gốc.

Bảng `fireant_financial_data_general` lọc `Quarter = 0` là dòng **năm** (không phải quý).
Ngoài `ShareAtPeriodEnd` còn lấy `PB` và `PriceAtPeriodEnd` để đối chiếu chéo ở mục E và F.

In [2]:
bm = (pd.read_csv(f"{RESULTS}/book_to_market_panel.csv")
        .set_index(["symbol", "period"]).sort_index())

GENERAL_SQL = """
select symbol, Year as period, PaidInCapital, ShareAtPeriodEnd, TreasuryStock,
       PriceAtPeriodEnd, MarketCapAtPeriodEnd, PB, BookValuePerShare
from fireant_financial_data_general
where Quarter = 0
"""
with closing(sqlite3.connect(DB)) as conn:
    general = pd.read_sql(GENERAL_SQL, conn)
general = general.set_index(["symbol", "period"]).sort_index()

pd.Series({
    "dòng panel BM": len(bm),
    "  trong đó có BM": int(bm["bm"].notna().sum()),
    "dòng năm trong fireant_financial_data_general": len(general),
    "  số mã": general.index.get_level_values("symbol").nunique(),
    "  khoảng năm": f'{general.index.get_level_values("period").min()}'
                    f'..{general.index.get_level_values("period").max()}',
    "  thiếu ShareAtPeriodEnd": int(general["ShareAtPeriodEnd"].isna().sum()),
})

dòng panel BM                                         21233
  trong đó có BM                                      17414
dòng năm trong fireant_financial_data_general         23671
  số mã                                                1848
  khoảng năm                                     2009..2025
  thiếu ShareAtPeriodEnd                                240
dtype: object

## A. `ShareAtPeriodEnd` có phải chỉ là `PaidInCapital/10.000` chép lại?

Phép so sánh này chạy **hoàn toàn bên trong bảng FireAnt** — cả hai cột đều lấy từ đó, nên
không có sai lệch do khác nguồn hay khác kỳ. Nếu hai cột trùng nhau ở mọi dòng thì
`ShareAtPeriodEnd` không mang thêm thông tin gì và notebook này dừng ở đây.

Dùng `PaidInCapital` của chính FireAnt (không phải `paid_in_capital` trong extract đã hiệu
chỉnh) là cố ý: mục này hỏi về **bảng nguồn**, không hỏi về pipeline.

In [3]:
base = general.dropna(subset=["PaidInCapital", "ShareAtPeriodEnd"])
base = base[(base["PaidInCapital"] > 0) & (base["ShareAtPeriodEnd"] > 0)].copy()

base["implied"] = base["PaidInCapital"] / PAR_VALUE_VND
# Lệch tương đối, có dấu: âm = lưu hành ít hơn phát hành (dấu hiệu cổ phiếu quỹ),
# dương = lưu hành nhiều hơn phát hành, điều không thể xảy ra về mặt định nghĩa.
base["rel"] = (base["ShareAtPeriodEnd"] - base["implied"]) / base["implied"]
base["has_treasury"] = base["TreasuryStock"].fillna(0).ne(0)

# "Khớp" đo bằng cổ phiếu chứ không bằng phần trăm: chênh dưới 1 cổ phiếu là làm tròn,
# không phải sự kiện doanh nghiệp.
diff_shares = base["ShareAtPeriodEnd"] - base["implied"]
exact = diff_shares.abs() < 1

pd.Series({
    "dòng so sánh được": len(base),
    "khớp (chênh < 1 cổ phiếu)": int(exact.sum()),
    "  tỷ lệ": f"{100 * exact.mean():.1f}%",
    "lưu hành < phát hành": int((diff_shares <= -1).sum()),
    "lưu hành > phát hành (bất khả thi)": int((diff_shares >= 1).sum()),
})

dòng so sánh được                     23290
khớp (chênh < 1 cổ phiếu)             17008
  tỷ lệ                               73.0%
lưu hành < phát hành                   5456
lưu hành > phát hành (bất khả thi)      826
dtype: object

In [4]:
# Phân vị của độ lệch. Đuôi trái tới -99% và đuôi phải tới +118 lần: cả hai đều vượt xa
# mọi mức cổ phiếu quỹ hợp lý, nên bảng này đã cho thấy có ít nhất hai hiện tượng khác nhau
# đang trộn vào nhau.
base["rel"].describe([.001, .01, .05, .25, .5, .75, .95, .99, .999]).round(5)

count    23290.00000
mean         0.04587
std          1.37901
min         -0.99467
0.1%        -0.89641
1%          -0.19868
5%          -0.04350
25%          0.00000
50%          0.00000
75%          0.00000
95%          0.00000
99%          0.97959
99.9%        6.88003
max        118.38873
Name: rel, dtype: float64

## B. Chênh lệch có phải là cổ phiếu quỹ?

Cổ phiếu quỹ ghi theo **giá vốn**, không quy ra số cổ phiếu ở mệnh giá được, nên không thể
đối chiếu từng đồng. Nhưng dấu thì đối chiếu được: nếu chênh lệch đúng là cổ phiếu quỹ, mọi
dòng có `rel < 0` phải có `TreasuryStock != 0`, và các dòng `rel > 0` thì không.

Chia `rel` thành năm khoảng và đếm chéo với sự tồn tại của cổ phiếu quỹ.

In [5]:
bucket = pd.cut(base["rel"], [-np.inf, -MAX_TREASURY, -TOL_EQUAL, TOL_EQUAL, MAX_TREASURY, np.inf],
                labels=["< -30%", "-30%..-0,5%", "khớp ±0,5%", "+0,5%..+30%", "> +30%"])

pd.DataFrame({
    "dòng": base.groupby(bucket, observed=False).size(),
    "có cổ phiếu quỹ": base.groupby(bucket, observed=False)["has_treasury"].sum(),
    "tỷ lệ có cổ phiếu quỹ": (base.groupby(bucket, observed=False)["has_treasury"].mean() * 100).round(1),
})

,dòng,có cổ phiếu quỹ,tỷ lệ có cổ phiếu quỹ
rel,,,
< -30%,174,42,24.1
"-30%..-0,5%",2662,2441,91.7
"khớp ±0,5%",19686,2963,15.1
"+0,5%..+30%",297,34,11.4
> +30%,471,46,9.8


Khoảng **−30%..−0,5%** là khoảng duy nhất mà cổ phiếu quỹ giải thích được gần như trọn
vẹn. Hai khoảng đuôi có tỷ lệ cổ phiếu quỹ *thấp hơn* cả nhóm khớp — chúng không phải hiện
tượng kế toán mà là lỗi dữ liệu, mục C xử lý.

Hai chuỗi thời gian dưới đây cho thấy dạng "sạch" của chênh lệch: vốn điều lệ tăng theo bậc
thang, số lưu hành bám sát nhưng thấp hơn đúng phần cổ phiếu quỹ, rồi trùng khít trở lại
khi cổ phiếu quỹ bị hủy hoặc bán ra.

Một chi tiết dễ bỏ sót trong bảng trên: nhóm **khớp ±0,5%** vẫn chứa gần 3.000 dòng *có*
cổ phiếu quỹ. Tức FireAnt không phải lúc nào cũng trừ — với nhiều firm-year, `ShareAtPeriodEnd`
chính là số phát hành dù bảng cân đối có ghi cổ phiếu quỹ.

Hệ quả: đổi sang `ShareAtPeriodEnd` **vá được một phần**, không phải toàn bộ, vấn đề cổ
phiếu quỹ. Cell dưới đo phần được vá.

In [6]:
ts = base[base["has_treasury"]]
netted = ts["ShareAtPeriodEnd"] < ts["implied"] - 1

pd.Series({
    "dòng năm có ghi cổ phiếu quỹ": len(ts),
    "  vendor CÓ trừ (outstanding < issued)": f"{int(netted.sum())} ({100 * netted.mean():.1f}%)",
    "  vendor KHÔNG trừ (bằng nhau)": int((ts["ShareAtPeriodEnd"].sub(ts["implied"]).abs() < 1).sum()),
    "  outstanding > issued (lỗi)": int((ts["ShareAtPeriodEnd"] > ts["implied"] + 1).sum()),
    "trung vị mức trừ, khi có trừ": f"{-100 * ts.loc[netted, 'rel'].median():.2f}%",
})

dòng năm có ghi cổ phiếu quỹ                      5526
  vendor CÓ trừ (outstanding < issued)    4977 (90.1%)
  vendor KHÔNG trừ (bằng nhau)                     459
  outstanding > issued (lỗi)                        84
trung vị mức trừ, khi có trừ                     0.50%
dtype: object

In [7]:
def trace(symbol: str) -> pd.DataFrame:
    """Vốn điều lệ, số lưu hành và khoảng cách giữa chúng, theo năm, cho một mã."""
    s = base.xs(symbol, level="symbol")
    return pd.DataFrame({
        "paid_in_capital": s["PaidInCapital"],
        "implied": s["implied"].astype("int64"),
        "outstanding": s["ShareAtPeriodEnd"].astype("int64"),
        "chênh (cp)": (s["implied"] - s["ShareAtPeriodEnd"]).round(0).astype("int64"),
        "chênh %": (-100 * s["rel"]).round(3),
        "có CP quỹ": s["has_treasury"],
    })

trace("VNM")

,paid_in_capital,implied,outstanding,chênh (cp),chênh %,có CP quỹ
period,,,,,,
2009,3.512653e+12,351265300,351249980,15320,0.004,True
2010,3.530721e+12,353072120,353006100,66020,0.019,True
2011,5.561148e+12,556114754,555867614,247140,0.044,True
2012,8.339558e+12,833955796,833525676,430120,0.052,True
2013,8.339558e+12,833955796,833467061,488735,0.059,True
2014,1.000641e+13,1000641399,1000118604,522795,0.052,True
2015,1.200662e+13,1200662193,1200139398,522795,0.044,True
2016,1.451453e+13,1451453429,1451426329,27100,0.002,True
2017,1.451453e+13,1451453429,1451278520,174909,0.012,True


In [8]:
trace("HPG")

,paid_in_capital,implied,outstanding,chênh (cp),chênh %,có CP quỹ
period,,,,,,
2009,1.963640e+12,196363998,196363998,0,-0.000,False
2010,3.178498e+12,317849760,317849760,0,-0.000,False
2011,3.178498e+12,317849760,313618830,4230930,1.331,True
2012,4.190525e+12,419052533,419052533,0,-0.000,False
2013,4.190525e+12,419052533,419052533,0,-0.000,False
2014,4.819082e+12,481908175,481908175,0,-0.000,False
2015,7.329514e+12,732951419,732880219,71200,0.010,True
2016,8.428750e+12,842874956,842765656,109300,0.013,True
2017,1.517079e+13,1517079000,1516909673,169327,0.011,True


## C. Phía bất khả thi: lưu hành > phát hành

Số cổ phiếu lưu hành không thể vượt số đã phát hành. Mọi dòng `rel > 0` đều là lỗi vendor,
và độ lớn của chúng (tới hơn 100 lần) loại trừ khả năng làm tròn.

Giả thuyết: FireAnt điền số cổ phiếu **hiện tại** cho toàn bộ lịch sử của một số mã. Kiểm
tra được: tìm những mã có `ShareAtPeriodEnd` đứng yên suốt nhiều năm trong khi vốn điều lệ
thay đổi. Nếu đúng, dùng cột này thô là nhét look-ahead vào panel.

In [9]:
per_symbol = base.groupby(level="symbol").agg(
    năm=("ShareAtPeriodEnd", "size"),
    số_giá_trị_outstanding=("ShareAtPeriodEnd", "nunique"),
    số_giá_trị_implied=("implied", "nunique"),
)
long_enough = per_symbol[per_symbol["năm"] >= 5]
frozen = long_enough[(long_enough["số_giá_trị_outstanding"] == 1)
                     & (long_enough["số_giá_trị_implied"] > 1)]

pd.Series({
    "mã có >= 5 năm dữ liệu": len(long_enough),
    "  outstanding đứng yên trong khi vốn điều lệ đổi": len(frozen),
    "  tỷ lệ": f"{100 * len(frozen) / len(long_enough):.1f}%",
    "dòng năm thuộc các mã đó": int(base.index.get_level_values("symbol").isin(frozen.index).sum()),
})

mã có >= 5 năm dữ liệu                              1713
  outstanding đứng yên trong khi vốn điều lệ đổi     140
  tỷ lệ                                             8.2%
dòng năm thuộc các mã đó                            1524
dtype: object

In [10]:
# Ví dụ rõ nhất: cùng một con số lặp lại qua nhiều năm, đúng bằng số cổ phiếu của mã đó ở
# thời điểm crawl, trong khi vốn điều lệ của những năm đó nhỏ hơn nhiều lần.
trace("VEF")

,paid_in_capital,implied,outstanding,chênh (cp),chênh %,có CP quỹ
period,,,,,,
2011,1.040144e+11,10401437,166601050,-156199612,-1501.712,False
2012,1.046376e+11,10463760,166601050,-156137289,-1492.172,False
2013,1.057990e+11,10579900,166601050,-156021149,-1474.694,False
2014,1.057990e+11,10579900,166601050,-156021149,-1474.694,False
2015,1.666040e+12,166604050,166601050,3000,0.002,False
2016,1.666040e+12,166604050,166601050,3000,0.002,False
2017,1.666040e+12,166604050,166601050,3000,0.002,False
2018,1.666040e+12,166604050,166601050,3000,0.002,True
2019,1.666040e+12,166604050,166601050,3000,0.002,True


In [11]:
# Xếp hạng các dòng lệch dương lớn nhất, kèm cột đánh dấu mã có bị đóng băng hay không.
worst_positive = base[base["rel"] > TOL_EQUAL].copy()
worst_positive["đóng băng"] = worst_positive.index.get_level_values("symbol").isin(frozen.index)
worst_positive["lần"] = (worst_positive["ShareAtPeriodEnd"] / worst_positive["implied"]).round(2)

print(f'{len(worst_positive)} dòng có lưu hành > phát hành quá 0,5%; '
      f'{int(worst_positive["đóng băng"].sum())} trong số đó thuộc mã bị đóng băng')
worst_positive.nlargest(15, "lần")[["PaidInCapital", "implied", "ShareAtPeriodEnd",
                                    "lần", "has_treasury", "đóng băng"]]

768 dòng có lưu hành > phát hành quá 0,5%; 270 trong số đó thuộc mã bị đóng băng


PaidInCapital       implied  ShareAtPeriodEnd     lần  has_treasury  đóng băng
symbol period                                                                                
HPA    2011     2.094000e+10  2.094000e+06       250000000.0  119.39          True      False
       2012     2.094000e+10  2.094000e+06       250000000.0  119.39          True      False
GQN    2017     1.344911e+08  1.344911e+04          940000.0   69.89         False      False
       2018     1.344911e+08  1.344911e+04          940000.0   69.89         False      False
NNQ    2018     3.046878e+08  3.046878e+04         1388424.0   45.57         False      False
DTE    2019     1.378000e+10  1.378000e+06        38405640.0   27.87         False      False
VEF    2011     1.040144e+11  1.040144e+07       166601050.0   16.02         False       True
       2012     1.046376e+11  1.046376e+07       166601050.0   15.92         False       True
       2013     1.057990e+11  1.057990e+07       166601050.0   15.75         False       True
       2014     1.057990e+11  1.057990e+07       166601050.0   15.75         False       True
VNB    2014     4.602496e+10  4.602496e+06        67909960.0   14.76         False      False
       2015     4.602496e+10  4.602496e+06        67909960.0   14.76         False      False
MXC    2017     2.239582e+09  2.239582e+05         2981300.0   13.31         False      False
       2018     2.239582e+09  2.239582e+05         2981300.0   13.31         False      False
       2019     2.239582e+09  2.239582e+05         2981300.0   13.31         False      False

Bài kiểm tra "đóng băng" ở trên chỉ bắt được mã có `nunique == 1` trên toàn bộ lịch sử.
Một mã bị đóng băng **một phần** — vài năm đầu dùng số hiện tại rồi mới đúng — sẽ lọt lưới
nếu độ lệch rơi vào khoảng −30%..0, tức trà trộn vào đúng vùng mà cổ phiếu quỹ chiếm đa số.

Lưới thứ hai bắt phần còn lại: số lưu hành chỉ được **giảm** khi doanh nghiệp mua cổ phiếu
quỹ. Giảm mà bảng cân đối không ghi cổ phiếu quỹ nào là dấu hiệu vendor sai, không phải sự
kiện doanh nghiệp.

In [12]:
ordered = base.sort_index()
prev_out = ordered.groupby(level="symbol")["ShareAtPeriodEnd"].shift(1)
prev_year = pd.Series(ordered.index.get_level_values("period"), index=ordered.index) \
              .groupby(level="symbol").shift(1)

contiguous = (pd.Series(ordered.index.get_level_values("period"), index=ordered.index) - prev_year).eq(1)
dropped = contiguous & (ordered["ShareAtPeriodEnd"] < prev_out * (1 - TOL_EQUAL))

pd.Series({
    "cặp năm liền kề so sánh được": int(contiguous.sum()),
    "số lưu hành giảm > 0,5%": int(dropped.sum()),
    "  có ghi cổ phiếu quỹ (hợp lý)": int((dropped & ordered["has_treasury"]).sum()),
    "  KHÔNG ghi cổ phiếu quỹ (đáng ngờ)": int((dropped & ~ordered["has_treasury"]).sum()),
})

cặp năm liền kề so sánh được           21330
số lưu hành giảm > 0,5%                  339
  có ghi cổ phiếu quỹ (hợp lý)           282
  KHÔNG ghi cổ phiếu quỹ (đáng ngờ)       57
dtype: int64

## D. Ảnh hưởng lên BM

Dựng lại BM trên đúng panel hiện có, chỉ thay mẫu số. Book equity và giá giữ nguyên, nên
mọi chênh lệch quan sát được đều đến từ cách đếm cổ phiếu.

Tỷ số `bm_out / bm` lớn hơn 1 nghĩa là BM đang dùng **đang bị hạ thấp** — cổ phiếu trông
đắt hơn thực tế, và bị đẩy ra khỏi nhóm value.

Chiều của sai lệch quan trọng hơn độ lớn trung vị: nó luôn cùng một dấu, và chỉ tác động
lên nhóm doanh nghiệp có mua cổ phiếu quỹ. Đó là một thiên lệch có hệ thống theo đặc điểm
doanh nghiệp, không phải nhiễu ngẫu nhiên.

In [13]:
panel = bm.join(general[["ShareAtPeriodEnd", "PB", "PriceAtPeriodEnd"]], how="left")
panel = panel[panel["bm"].notna()].copy()

panel["shares_out"] = panel["ShareAtPeriodEnd"].where(panel["ShareAtPeriodEnd"] > 0)
panel["rel"] = (panel["shares_out"] - panel["shares_issued"]) / panel["shares_issued"]
panel["bm_out"] = (panel["book_equity"] / (panel["close"] * panel["shares_out"])) \
                    .where(panel["book_equity"] > 0)

comparable = panel.dropna(subset=["bm", "bm_out"])
ratio = comparable["bm_out"] / comparable["bm"]

pd.Series({
    "dòng có BM": len(panel),
    "  so sánh được hai cách đếm": len(comparable),
    "BM đổi > 1%": f'{int((ratio.sub(1).abs() > 0.01).sum())} '
                   f'({100 * (ratio.sub(1).abs() > 0.01).mean():.1f}%)',
    "BM đổi > 5%": f'{int((ratio.sub(1).abs() > 0.05).sum())} '
                   f'({100 * (ratio.sub(1).abs() > 0.05).mean():.1f}%)',
    "BM đổi > 20%": f'{int((ratio.sub(1).abs() > 0.20).sum())} '
                    f'({100 * (ratio.sub(1).abs() > 0.20).mean():.1f}%)',
})

dòng có BM                          17414
  so sánh được hai cách đếm         17351
BM đổi > 1%                    331 (1.9%)
BM đổi > 5%                    294 (1.7%)
BM đổi > 20%                   231 (1.3%)
dtype: object

In [14]:
pd.DataFrame({
    "tất cả": ratio.describe([.01, .05, .25, .5, .75, .95, .99]),
    "có cổ phiếu quỹ": ratio[comparable["has_treasury"]].describe([.01, .05, .25, .5, .75, .95, .99]),
    "không có": ratio[~comparable["has_treasury"]].describe([.01, .05, .25, .5, .75, .95, .99]),
}).round(4)

,tất cả,có cổ phiếu quỹ,không có
count,17351.0000,4537.0000,12814.0000
mean,1.0900,1.0206,1.1146
std,3.5454,0.5246,4.1135
min,0.1540,0.4888,0.1540
1%,0.9091,0.9992,0.8710
5%,1.0000,1.0000,1.0000
25%,1.0000,1.0000,1.0000
50%,1.0000,1.0000,1.0000
75%,1.0000,1.0000,1.0000
95%,1.0000,1.0000,1.0000


In [15]:
# Những dòng bị ảnh hưởng nặng nhất. Cần nhìn tận mắt để phân biệt hai loại: doanh nghiệp
# giữ nhiều cổ phiếu quỹ thật (sửa là đúng) và dòng lỗi vendor (sửa là sai).
top = comparable.assign(tỷ_số=ratio).nlargest(15, "tỷ_số")
# Cột này nói trước kết luận của mục G: gần như toàn bộ đuôi cực đoan là lỗi vendor và sẽ
# bị loại, nên đừng đọc bảng này như "mức sửa mà quy tắc sẽ áp".
top["quy_tắc_giữ"] = top["rel"].le(TOL_EQUAL) & top["rel"].ge(-MAX_TREASURY)
top[["shares_issued", "shares_out", "rel", "bm", "bm_out", "tỷ_số",
     "has_treasury", "quy_tắc_giữ"]].round(4)

shares_issued  shares_out     rel      bm    bm_out     tỷ_số  has_treasury  quy_tắc_giữ
symbol period                                                                                          
JVC    2017      112500171.0    600000.0 -0.9947  1.1465  214.9718  187.5003         False        False
       2023      112500171.0    600000.0 -0.9947  1.3127  246.1288  187.5003         False        False
       2018      112500171.0    600000.0 -0.9947  1.5608  292.6597  187.5003         False        False
       2022      112500171.0    600000.0 -0.9947  1.3013  243.9972  187.5003         False        False
       2024      112500171.0    600000.0 -0.9947  1.2618  236.5927  187.5003         False        False
       2020      112500171.0    600000.0 -0.9947  0.7076  132.6804  187.5003         False        False
KDH    2015      180000000.0   3200000.0 -0.9822  0.8395   47.2241   56.2500         False        False
S99    2013       12496929.0    300000.0 -0.9760  1.8554   77.2908   41.6564         False        False
       2014       12496929.0    300000.0 -0.9760  1.1375   47.3838   41.6564         False        False
KBC    2024      767604759.0  29570000.0 -0.9615  0.8888   23.0732   25.9589         False        False
SJS    2015      100000000.0   5000000.0 -0.9500  0.8412   16.8244   20.0000          True        False
       2016      100000000.0   5000000.0 -0.9500  0.8923   17.8451   20.0000          True        False
       2017      100000000.0   5000000.0 -0.9500  0.7475   14.9491   20.0000          True        False
EVG    2022      215249836.0  18000000.0 -0.9164  3.1500   37.6685   11.9583         False        False
       2024      215249836.0  18000000.0 -0.9164  1.6885   20.1921   11.9583         False        False

## E. Đối chiếu chéo với `1/PB` của FireAnt

FireAnt công bố `PB` của riêng nó. Nếu BM dựng theo một cách đếm khớp `1/PB` còn cách kia
thì không, ta biết FireAnt dùng cách nào.

**Giới hạn cần ghi rõ khi viết luận văn:** `PB` của FireAnt tự nó được dựng từ
`BookValuePerShare × ShareAtPeriodEnd`, nên phép này chứng minh *nhất quán với vendor*, chứ
không phải đúng với thực tế. Bằng chứng độc lập vẫn là tương quan với cổ phiếu quỹ ở mục B.

Chỉ so trên các dòng mà hai cách đếm thực sự khác nhau — 73% dòng còn lại trùng nhau và sẽ
làm loãng kết quả thành 1,0000 ở cả hai cột.

In [16]:
cross = panel[(panel["PB"] > 0) & panel["shares_out"].notna()].copy()
disagree = cross[cross["rel"].abs() > TOL_EQUAL]
inv_pb = 1 / disagree["PB"]

a = disagree["bm"] / inv_pb                                                    # dùng số phát hành
b = (disagree["book_equity"] / (disagree["close"] * disagree["shares_out"])) / inv_pb  # dùng số lưu hành

pd.DataFrame({
    "median tỷ số với 1/PB": [a.median(), b.median()],
    "dòng lệch > 1%": [int(a.sub(1).abs().gt(0.01).sum()), int(b.sub(1).abs().gt(0.01).sum())],
    "dòng lệch > 5%": [int(a.sub(1).abs().gt(0.05).sum()), int(b.sub(1).abs().gt(0.05).sum())],
}, index=[f"BM dùng shares_issued (n={len(disagree)})",
          f"BM dùng shares_outstanding (n={len(disagree)})"]).round(4)

,median tỷ số với 1/PB,dòng lệch > 1%,dòng lệch > 5%
BM dùng shares_issued (n=2467),1.0,366,311
BM dùng shares_outstanding (n=2467),1.0,53,19


## F. Kiểm tra giá — không liên quan tới số cổ phiếu, nhưng gần như miễn phí

`PriceAtPeriodEnd` là giá cuối kỳ của chính FireAnt. So với `close` mà
`book_to_market_calculation.ipynb` tự dựng (phiên có khớp lệnh cuối cùng của tháng 12, từ
bảng daily) sẽ kiểm chứng luôn phần SQL gom month-end.

In [17]:
price_ratio = (panel["close"] / panel["PriceAtPeriodEnd"]) \
                .replace([np.inf, -np.inf], np.nan).dropna()

pd.Series({
    "dòng so sánh được": len(price_ratio),
    "median tỷ số": round(price_ratio.median(), 4),
    "khớp trong ±0,1%": f"{100 * price_ratio.sub(1).abs().lt(0.001).mean():.1f}%",
    "khớp trong ±1%": f"{100 * price_ratio.sub(1).abs().lt(0.01).mean():.1f}%",
    "lệch > 5%": int(price_ratio.sub(1).abs().gt(0.05).sum()),
})

dòng so sánh được    17351
median tỷ số           1.0
khớp trong ±0,1%     97.6%
khớp trong ±1%       98.2%
lệch > 5%              123
dtype: object

## G. Quy tắc đề xuất

Không nguồn nào sạch một mình: số phát hành đếm thừa cổ phiếu quỹ, số lưu hành thì có dòng
bị đóng băng. Lai hai nguồn với hai chốt chặn:

- **`rel > +0,5%`** → loại. Lưu hành không thể nhiều hơn phát hành; đây là lỗi vendor.
- **`rel < −30%`** → loại. Vượt quá mọi mức cổ phiếu quỹ còn tin được; nhiều khả năng là
  đóng băng một phần.

Dòng bị loại rơi về `shares_issued`, tức về đúng hành vi hiện tại của notebook — nên thay
đổi này không làm mất dòng nào khỏi panel.

Trong dung sai `+0,5%` thì vẫn dùng số vendor nhưng **kẹp trần tại số phát hành**: lưu hành
vượt phát hành là nhiễu khi nhỏ và là lỗi khi lớn, không trường hợp nào là số cổ phiếu thật.
Nhờ vậy tác động lên BM một chiều tuyệt đối — chỉ tăng, không bao giờ giảm — nên kiểm toán
được. Đây đúng là quy tắc đã áp vào `book_to_market_calculation.ipynb`.

In [18]:
usable = (panel["shares_out"].notna()
          & panel["rel"].le(TOL_EQUAL)
          & panel["rel"].ge(-MAX_TREASURY))

# Kẹp trần tại số phát hành: lệch dương nhỏ trong dung sai là nhiễu làm tròn, không phải
# số cổ phiếu. Không kẹp thì 50 dòng bị hạ BM tới 0,5% mà không tương ứng với điều gì cả.
panel["shares_final"] = (panel["shares_out"].clip(upper=panel["shares_issued"])
                         .where(usable, panel["shares_issued"]))
panel["shares_source"] = np.where(usable, "outstanding", "par_implied")
panel["bm_final"] = (panel["book_equity"] / (panel["close"] * panel["shares_final"])) \
                      .where(panel["book_equity"] > 0)

pd.Series({
    "dòng có BM": len(panel),
    "dùng ShareAtPeriodEnd": f"{int(usable.sum())} ({100 * usable.mean():.1f}%)",
    "  thực sự khác par-implied": int((usable & panel["rel"].lt(-TOL_EQUAL)).sum()),
    "rơi về par-implied": int((~usable).sum()),
    "  thiếu ShareAtPeriodEnd": int(panel["shares_out"].isna().sum()),
    "  lưu hành > phát hành": int((panel["shares_out"].notna() & panel["rel"].gt(TOL_EQUAL)).sum()),
    "  lệch < -30%": int((panel["shares_out"].notna() & panel["rel"].lt(-MAX_TREASURY)).sum()),
    "BM mất đi do quy tắc": int(panel["bm"].notna().sum() - panel["bm_final"].notna().sum()),
})

dòng có BM                            17414
dùng ShareAtPeriodEnd         17013 (97.7%)
  thực sự khác par-implied             2129
rơi về par-implied                      401
  thiếu ShareAtPeriodEnd                 63
  lưu hành > phát hành                  233
  lệch < -30%                           105
BM mất đi do quy tắc                      0
dtype: object

In [19]:
# Trung vị theo năm, trước và sau. Điều cần nhìn không phải độ lớn mà là có drift theo thời
# gian hay không: nếu mức thay đổi tăng dần về quá khứ thì nó đụng tới kết luận về value
# premium; nếu rải đều thì nó chỉ đụng tới thứ hạng cross-section — vốn là cái quyết định
# danh mục.
by_year = pd.DataFrame({
    "dòng": panel["bm"].groupby(level="period").size(),
    "bm_cũ": panel["bm"].groupby(level="period").median(),
    "bm_mới": panel["bm_final"].groupby(level="period").median(),
    "dòng đổi > 1%": ((panel["bm_final"] / panel["bm"] - 1).abs() > 0.01)
                       .groupby(level="period").sum(),
})
by_year["% thay đổi trung vị"] = (100 * (by_year["bm_mới"] / by_year["bm_cũ"] - 1))
by_year.round(4)

,dòng,bm_cũ,bm_mới,dòng đổi > 1%,% thay đổi trung vị
period,,,,,
2009,274,0.6970,0.6970,0,0.0
2010,544,0.9193,0.9193,0,0.0
2011,672,1.9180,1.9180,0,0.0
2012,719,1.8132,1.8132,0,0.0
2013,708,1.5218,1.5218,0,0.0
2014,713,1.2215,1.2215,0,0.0
2015,799,1.1323,1.1323,0,0.0
2016,941,1.0945,1.0945,0,0.0
2017,1232,1.0633,1.0633,0,0.0


In [20]:
# Độ nhạy của ngưỡng -30%. Nếu số dòng dùng được gần như không đổi trong khoảng 20-50% thì
# ngưỡng không phải là tham số cần bảo vệ trong luận văn.
rows = []
for thr in [0.10, 0.20, 0.30, 0.40, 0.50, 1.00]:
    u = panel["shares_out"].notna() & panel["rel"].le(TOL_EQUAL) & panel["rel"].ge(-thr)
    sh = panel["shares_out"].clip(upper=panel["shares_issued"]).where(u, panel["shares_issued"])
    b = (panel["book_equity"] / (panel["close"] * sh)).where(panel["book_equity"] > 0)
    rows.append({
        "ngưỡng": f"-{thr:.0%}",
        "dùng outstanding": int(u.sum()),
        "median BM": round(b.median(), 5),
        "dòng đổi > 1%": int(((b / panel["bm"] - 1).abs() > 0.01).sum()),
    })
pd.DataFrame(rows).set_index("ngưỡng")

,dùng outstanding,median BM,dòng đổi > 1%
ngưỡng,,,
-10%,16828,1.09785,185
-20%,16985,1.09904,28
-30%,17013,1.09963,0
-40%,17019,1.09994,6
-50%,17024,1.10007,11
-100%,17118,1.10439,105


## H. Xuất file để soi tay

Một dòng cho mỗi firm-year có BM, mang cả hai cách đếm, cả hai BM, và lý do chọn. Đây là
file để mở bằng mắt, không phải input của pipeline — merge vào
`book_to_market_calculation.ipynb` là bước sau.

In [21]:
audit = panel[["book_equity", "close", "close_date", "price_stale", "has_treasury",
               "shares_issued", "shares_out", "rel", "shares_source", "shares_final",
               "bm", "bm_out", "bm_final", "PB"]].copy()
audit["bm_ratio"] = audit["bm_final"] / audit["bm"]

out_path = f"{RESULTS}/share_count_diagnostics.csv"
audit.to_csv(out_path)
print(out_path, audit.shape)
audit.head()

../data/preprocessing_pipeline_results/share_count_diagnostics.csv (17414, 15)


book_equity    close  close_date price_stale  has_treasury  shares_issued  shares_out  rel shares_source  shares_final        bm    bm_out  bm_final       PB  bm_ratio
symbol period                                                                                                                                                                          
A32    2018    2.008266e+11  30200.0  2018-12-21       False         False      6800000.0   6800000.0  0.0   outstanding     6800000.0  0.977925  0.977925  0.977925  1.02265       1.0
       2019    2.236159e+11  28000.0  2019-12-24       False         False      6800000.0   6800000.0  0.0   outstanding     6800000.0  1.174454  1.174454  1.174454  0.85147       1.0
       2020    2.422229e+11  34500.0  2020-12-31       False         False      6800000.0   6800000.0  0.0   outstanding     6800000.0  1.032493  1.032493  1.032493  0.96855       1.0
       2021    2.380572e+11  31800.0  2021-12-31       False         False      6800000.0   6800000.0  0.0   outstanding     6800000.0  1.100893  1.100893  1.100893  0.90710       1.0
       2022    2.153997e+11  32000.0  2022-12-01       False         False      6800000.0   6800000.0  0.0   outstanding     6800000.0  0.989888  0.989888  0.989888  0.95339       1.0

In [22]:
# Danh sách mã cần soi trước tiên: bị quy tắc loại, hoặc BM đổi trên 10%.
flagged = audit[(audit["shares_source"] == "par_implied") | (audit["bm_ratio"] > 1.10)]
print(f'{len(flagged)} dòng / {flagged.index.get_level_values("symbol").nunique()} mã')
flagged.sort_values("bm_ratio", ascending=False) \
       [["shares_issued", "shares_out", "rel", "has_treasury", "shares_source",
         "bm", "bm_final", "bm_ratio"]] \
       .head(25).round(4)

401 dòng / 236 mã


shares_issued    shares_out     rel  has_treasury shares_source      bm  bm_final  bm_ratio
symbol period                                                                                             
ONW    2018     2.000000e+06           NaN     NaN         False   par_implied  0.0033    0.0033       1.0
SSH    2021     2.500000e+08  3.750000e+08  0.5000         False   par_implied  0.1005    0.1005       1.0
THD    2020     5.390000e+07  3.500000e+08  5.4935         False   par_implied  0.2445    0.2445       1.0
IPH    2019     1.000000e+06  2.054950e+05 -0.7945         False   par_implied  0.1454    0.1454       1.0
VNH    2021     8.023071e+06           NaN     NaN         False   par_implied  0.0883    0.0883       1.0
KDH    2024     1.011143e+09  1.800000e+08 -0.8220         False   par_implied  0.4765    0.4765       1.0
MVN    2018     1.165549e+09  1.404606e+09  0.2051         False   par_implied  0.4890    0.4890       1.0
CAP    2017     4.760088e+06  5.000000e+05 -0.8950         False   par_implied  0.3964    0.3964       1.0
       2018     4.760088e+06  5.000000e+05 -0.8950         False   par_implied  0.4155    0.4155       1.0
MTP    2018     3.998367e+06  6.593767e+06  0.6491          True   par_implied  1.3494    1.3494       1.0
SGH    2010     1.766297e+06  1.776300e+06  0.0057         False   par_implied  0.2282    0.2282       1.0
       2009     1.766297e+06  1.776300e+06  0.0057         False   par_implied  0.2740    0.2740       1.0
MNC    2011     7.017130e+06  8.069307e+06  0.1499         False   par_implied  3.6235    3.6235       1.0
SC5    2012     1.362236e+07  1.498405e+07  0.1000          True   par_implied  1.6851    1.6851       1.0
VC2    2011     8.000000e+06  1.186450e+07  0.4831          True   par_implied  1.9633    1.9633       1.0
SDY    2015     4.500000e+06  1.500000e+06 -0.6667         False   par_implied  0.4633    0.4633       1.0
TTZ    2019     7.570444e+06  1.500000e+06 -0.8019         False   par_implied  3.9018    3.9018       1.0
CCT    2017     2.752812e+07  2.848000e+07  0.0346         False   par_implied  0.9555    0.9555       1.0
CNA    2021     1.802865e+06  3.415555e+06  0.8945         False   par_implied  0.2144    0.2144       1.0
TDP    2025     8.822225e+07  9.369957e+07  0.0621         False   par_implied  0.4363    0.4363       1.0
MCP    2022     1.507134e+07  9.830798e+06 -0.3477          True   par_implied  0.9622    0.9622       1.0
CCT    2021     2.752812e+07  2.848000e+07  0.0346         False   par_implied  0.9710    0.9710       1.0
PVX    2011     2.500000e+08  3.999970e+08  0.6000          True   par_implied  1.6211    1.6211       1.0
MCM    2020     6.680000e+07  1.100000e+08  0.6467         False   par_implied  0.1891    0.1891       1.0
TV1    2022     2.669132e+07  1.000000e+07 -0.6253         False   par_implied  0.9998    0.9998       1.0